# Step 2: Embeddings and the vector store
Three embedding models over the same 4,300 chunks in ChromaDB, with metadata (bank, document type, fiscal year, section, risk category, page, access). Retrieval strategies: top-k, MMR, hybrid (vector plus BM25, reciprocal rank fusion) and hybrid with a cross encoder reranker. The bank and fiscal year are read from the question and applied as filters.

In [1]:
import sys, json, pandas as pd
sys.path.insert(0, '../src')
pd.set_option('display.max_colwidth', 120)

In [2]:
res = json.load(open('../reports/retrieval_eval.json'))
pd.DataFrame([{'config': k, **{m: v[m] for m in ('precision_at_k', 'hit_at_1', 'right_filing')}} for k, v in res['results'].items()]).sort_values('precision_at_k', ascending=False)

,config,precision_at_k,hit_at_1,right_filing
3,minilm/rerank,0.733,0.733,1.0
2,minilm/hybrid,0.622,0.733,1.0
7,bge-small/rerank,0.622,0.733,1.0
6,bge-small/hybrid,0.556,0.733,1.0
0,minilm/topk,0.489,0.600,1.0
4,bge-small/topk,0.422,0.467,1.0
1,minilm/mmr,0.311,0.600,1.0
5,bge-small/mmr,0.311,0.467,1.0


Relevance is decided by an explicit rule per query (right bank, right year, the metric named in the passage), printed below, so the precision figures can be checked by anyone.

In [3]:
pd.DataFrame(res['rules'], columns=['query', 'bank', 'year', 'relevant if passage matches'])

,query,bank,year,relevant if passage matches
0,What was RBC's provision for credit losses in fiscal 2025?,RBC,2025,provision for credit losses|\bpcl\b
1,How much did TD set aside for credit losses in 2024?,TD,2024,provision for credit losses|\bpcl\b
2,What was BMO's Common Equity Tier 1 ratio at the end of fiscal 2025?,BMO,2025,common equity tier 1|cet1
3,What was Scotiabank's liquidity coverage ratio in 2025?,BNS,2025,liquidity coverage ratio|\blcr\b
4,How does CIBC describe climate-related risk in its 2025 MD&A?,CIBC,2025,climate
5,"What market risk measures, such as VaR, did RBC report for 2024?",RBC,2024,value at risk|\bvar\b
6,What are TD's top and emerging risks in fiscal 2025?,TD,2025,top and emerging|emerging risk
7,How did BMO describe cyber and information security risk in 2024?,BMO,2024,cyber|information security
8,What was Scotiabank's gross impaired loans figure in 2024?,BNS,2024,impaired loans|\bgil\b
9,What is CIBC's net stable funding ratio for 2024?,CIBC,2024,net stable funding|\bnsfr\b


In [4]:
from jarvis.retrieve import Retriever, scope_of
r = Retriever('minilm')
q = "What was RBC's CET1 ratio at the end of fiscal 2025?"
print(scope_of(q))
pd.DataFrame(r.search(q, strategy='hybrid', k=5))[['bank', 'fiscal_year', 'document_type', 'section', 'page', 'similarity']]

Scope(banks=['RBC'], years=[2025], doc_types=[], quarters=[], quarterly=False)


,bank,fiscal_year,document_type,section,page,similarity
0,RBC,2025,Annual Report,Regulatory capital and TLAC available | Table 64,191,0.4637
1,RBC,2025,Annual Report,Capital management,187,0.4070
2,RBC,2025,Annual Report,Overview,9,0.4195
3,RBC,2025,Annual Report,Regulatory capital and TLAC available | Table 64,195,0.4318
4,RBC,2025,Annual Information Form,General Development Of The Business,9,0.5343


## Local against hosted embeddings, after the quarterly reports were added
The same 15 queries over the full index (annual and quarterly), with OpenAI text-embedding-3-small beside the two local models. OpenAI is a little better on plain vector search, but with the cross encoder reranker the free local MiniLM still leads, so it stays the default (and no filing text leaves the machine for indexing). Indexing the 5.4M tokens with OpenAI cost about $0.11.

In [5]:
v2 = json.load(open('../reports/retrieval_eval_v2.json'))
t = pd.DataFrame([{'model': k.split('/')[0], 'strategy': k.split('/')[1], 'precision_at_3': v['precision_at_k'], 'hit_at_1': v['hit_at_1']} for k, v in v2['results'].items()])
t.pivot(index='strategy', columns='model', values='precision_at_3').round(2)

model,bge-small,minilm,openai
strategy,,,
hybrid,0.51,0.60,0.62
mmr,0.31,0.31,0.47
rerank,0.67,0.73,0.69
topk,0.42,0.49,0.51


Metadata filtering: the same question with and without the scope filter.

In [6]:
from jarvis.retrieve import Scope
pd.DataFrame(r.search('credit risk exposure commercial real estate', strategy='topk', k=5, scope=Scope(['RBC'], [], [])))[['bank', 'fiscal_year', 'section', 'page']]

,bank,fiscal_year,section,page
0,RBC,2024,Credit risk,27
1,RBC,2025,Credit risk,23
2,RBC,2025,Credit risk,26
3,RBC,2024,Credit risk,25
4,RBC,2024,Credit risk,26
